# ⚖️ Fusion Agent — Training & Evaluation Notebook

Trains the fusion meta-classifier (or optimises weights) using
predictions from all trained agents on the validation set.
Then performs a full system-level evaluation.

**Prerequisite:** Run `spectral_train.ipynb`, `prosodic_train.ipynb`,
and `linguistic_train.ipynb` first to produce `checkpoints/*/best.pt`.


In [ ]:
!pip install -q torch torchaudio librosa soundfile transformers openai-whisper \
    scikit-learn matplotlib seaborn tqdm pyyaml rich

import sys, os
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

## 1 · Configuration

In [ ]:
DATA_DIR       = './data/asvspoof5'
KAGGLE_DATASET = 'aniket202411001/asvspoof5-flac'
DATA_FACTOR    = 0.6
BATCH_SIZE     = 16
SR             = 16_000
CHUNK_DURATION = 3.0
CHUNK_OVERLAP  = 0.5
NUM_WORKERS    = 2
CKPT_DIR       = './checkpoints'
FUSION_MODE    = 'meta_classifier'   # 'weighted' or 'meta_classifier'
# Default weights (used if FUSION_MODE == 'weighted')
WEIGHTS = {'spectral': 0.35, 'prosodic': 0.20, 'linguistic': 0.45}
WHISPER_MODEL  = 'openai/whisper-small'
BERT_MODEL     = 'bert-base-uncased'
GPT2_MODEL     = 'gpt2'
print('Config loaded.')

## 2 · Load trained agent models

In [ ]:
import torch
import numpy as np
from pathlib import Path
from utils.training_utils import detect_device, auto_download_dataset, load_checkpoint
from preprocessor.dataloader import detect_dataset_structure

auto_download_dataset(DATA_DIR, KAGGLE_DATASET)
device, device_type = detect_device()
layout = detect_dataset_structure(DATA_DIR)

# ── Spectral ──────────────────────────────────────────────────────────────────
from spectral.spectral_model import SpectralModel
spectral_ckpt = Path(CKPT_DIR) / 'spectral' / 'best.pt'
spectral_model = SpectralModel(n_features=100).to(device)
spectral_state = load_checkpoint(str(spectral_ckpt))
if spectral_state: spectral_model.load_state_dict(spectral_state['model_state_dict'])
spectral_model.eval()

# ── Prosodic ──────────────────────────────────────────────────────────────────
from prosodic.prosodic_model import ProsodicModel
prosodic_ckpt = Path(CKPT_DIR) / 'prosodic' / 'best.pt'
prosodic_model = ProsodicModel(n_features=5).to(device)
prosodic_state = load_checkpoint(str(prosodic_ckpt))
if prosodic_state: prosodic_model.load_state_dict(prosodic_state['model_state_dict'])
prosodic_model.eval()

# ── Linguistic ────────────────────────────────────────────────────────────────
from linguistic.linguistic_model import LinguisticClassifier
from transformers import AutoTokenizer, GPT2Tokenizer, GPT2LMHeadModel
ling_ckpt = Path(CKPT_DIR) / 'linguistic' / 'best.pt'
ling_model = LinguisticClassifier(bert_name=BERT_MODEL).to(device)
ling_state = load_checkpoint(str(ling_ckpt))
if ling_state: ling_model.load_state_dict(ling_state['model_state_dict'])
ling_model.eval()

bert_tok = AutoTokenizer.from_pretrained(BERT_MODEL)
gpt2_tok = GPT2Tokenizer.from_pretrained(GPT2_MODEL)
gpt2_mod = GPT2LMHeadModel.from_pretrained(GPT2_MODEL).eval().to(device)

print('All agent models loaded.')

## 3 · Collect agent predictions on validation set

In [ ]:
from preprocessor.dataloader import ASVspoofDataset, get_dataloader
from linguistic.linguistic_model import extract_text_features
import json, time
from pathlib import Path
from preprocessor.audio_utils import load_audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Build val datasets for spectral & prosodic
spec_val_ds  = ASVspoofDataset(DATA_DIR, 'val', 'spectral',  DATA_FACTOR, CHUNK_DURATION, CHUNK_OVERLAP, SR, layout)
pros_val_ds  = ASVspoofDataset(DATA_DIR, 'val', 'prosodic',  DATA_FACTOR, CHUNK_DURATION, CHUNK_OVERLAP, SR, layout)
ling_val_ds  = ASVspoofDataset(DATA_DIR, 'val', 'linguistic', DATA_FACTOR, chunk_duration=CHUNK_DURATION, sr=SR, layout=layout)

spec_loader  = get_dataloader(spec_val_ds,  BATCH_SIZE, NUM_WORKERS, shuffle=False)
pros_loader  = get_dataloader(pros_val_ds,  BATCH_SIZE, NUM_WORKERS, shuffle=False)

def collect_scores(model, loader, device):
    scores, labels = [], []
    with torch.no_grad():
        for feats, lbls in loader:
            out = model(feats.to(device))
            scores.extend(out.cpu().numpy().tolist())
            labels.extend(lbls.numpy().tolist())
    return np.array(scores), np.array(labels)

print('Collecting spectral scores …')
spec_scores, labels = collect_scores(spectral_model, spec_loader, device)

print('Collecting prosodic scores …')
pros_scores, _ = collect_scores(prosodic_model, pros_loader, device)

# Aggregate chunk scores per file (average) to align with linguistic (file-level)
def aggregate_chunk_scores(dataset, scores_arr):
    """Average chunk scores back to file-level."""
    file_scores = {}
    for idx, (path, ci, label) in enumerate(dataset.items):
        key = str(path)
        if key not in file_scores:
            file_scores[key] = {'scores': [], 'label': label}
        file_scores[key]['scores'].append(scores_arr[idx])
    results = [(np.mean(v['scores']), v['label']) for v in file_scores.values()]
    return np.array([r[0] for r in results]), np.array([r[1] for r in results])

spec_file_scores, file_labels = aggregate_chunk_scores(spec_val_ds, spec_scores)
pros_file_scores, _           = aggregate_chunk_scores(pros_val_ds, pros_scores)

print(f'File-level samples: {len(file_labels)}')
print(f'Spectral mean score: {spec_file_scores.mean():.4f}')
print(f'Prosodic mean score: {pros_file_scores.mean():.4f}')

In [ ]:
# ── Linguistic scores (file-level) ───────────────────────────────────────────
ling_cache = Path('./cache/transcripts/val.json')

if ling_cache.exists():
    with open(ling_cache) as f:
        val_transcripts = json.load(f)
    print(f'Loaded {len(val_transcripts)} cached val transcripts.')
else:
    print('No cached transcripts found. Run linguistic_train.ipynb Phase 1 first.')
    val_transcripts = []

ling_scores_list = []
if val_transcripts:
    for i, rec in enumerate(val_transcripts):
        text = rec.get('transcript', '')
        hand = extract_text_features(text, gpt2_tok, gpt2_mod, device=str(device))
        hand_t = torch.from_numpy(hand).unsqueeze(0).to(device)
        enc  = bert_tok(text, return_tensors='pt', truncation=True, max_length=512, padding='max_length')
        ids  = enc['input_ids'].to(device)
        mask = enc['attention_mask'].to(device)
        with torch.no_grad():
            p = ling_model(ids, mask, hand_t).item()
        ling_scores_list.append(p)
        print(f'  Linguistic {i+1}/{len(val_transcripts)}   ', end='\r')

ling_file_scores = np.array(ling_scores_list) if ling_scores_list else np.zeros(len(file_labels))
print(f'\nLinguistic mean score: {ling_file_scores.mean():.4f}')

## 4 · Train / configure fusion & evaluate

In [ ]:
from fusion.decision_agent import DecisionAgent
from fusion.evaluation import evaluate_system, compute_eer, compute_auc

# Align arrays to minimum length (in case of off-by-one from chunking)
n = min(len(file_labels), len(spec_file_scores), len(pros_file_scores), len(ling_file_scores))
file_labels      = file_labels[:n]
spec_file_scores = spec_file_scores[:n]
pros_file_scores = pros_file_scores[:n]
ling_file_scores = ling_file_scores[:n]

scores_matrix = np.column_stack([spec_file_scores, pros_file_scores, ling_file_scores])

fusion_agent = DecisionAgent(mode=FUSION_MODE, weights=WEIGHTS)

if FUSION_MODE == 'meta_classifier':
    fusion_agent.train(scores_matrix, file_labels)
    fused = fusion_agent._clf.predict_proba(scores_matrix)[:, 1]
else:
    fused = np.array([
        fusion_agent.fuse({'spectral': s, 'prosodic': p, 'linguistic': l})
        for s, p, l in zip(spec_file_scores, pros_file_scores, ling_file_scores)
    ])

print(f'Fused mean score: {fused.mean():.4f}')

results = evaluate_system(
    scores_dict={'spectral': spec_file_scores, 'prosodic': pros_file_scores, 'linguistic': ling_file_scores},
    labels=file_labels,
    fusion_scores=fused,
)

# Save fusion agent
Path(CKPT_DIR + '/fusion').mkdir(parents=True, exist_ok=True)
fusion_agent.save(CKPT_DIR + '/fusion/fusion_agent.pkl')

import json
with open(CKPT_DIR + '/fusion/val_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved.')

## 5 · Score distribution plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=False)
agent_data = [
    ('Spectral',   spec_file_scores),
    ('Prosodic',   pros_file_scores),
    ('Linguistic', ling_file_scores),
    ('Fused',      fused),
]

for ax, (name, scores) in zip(axes, agent_data):
    bona_s = scores[file_labels == 0]
    spoof_s = scores[file_labels == 1]
    ax.hist(bona_s,  bins=40, alpha=0.7, color='#4C9EE8', label='bonafide', density=True)
    ax.hist(spoof_s, bins=40, alpha=0.7, color='#E8734C', label='spoof',    density=True)
    ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='threshold')
    ax.set(title=f'{name} Score Distribution', xlabel='Score', ylabel='Density')
    ax.legend(fontsize=8)

plt.tight_layout()
out_path = CKPT_DIR + '/fusion/score_distributions.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')